In [ ]:
import gymnasium as gym
from h1 import h1
import numpy as np
import torch
import humanoid_bench

In [ ]:
env = gym.make(
        "h1-maze-v0",
        render_mode="rgb_array",
)
env.reset()
data = env.unwrapped.named.data

joint_positions = h1.fk_joint_positions(h1.body_tree["pelvis"], torch.from_numpy(np.array(data.qpos)).unsqueeze(0).float())

print(joint_positions)

In [ ]:
print(data.xanchor)

In [ ]:
# Build mapping from joint order to xanchor indices (assumed order)
joint_to_xanchor_mapping = {
    "free_base": 0,
    "left_hip_yaw": 1,
    "left_hip_roll": 2,
    "left_hip_pitch": 3,
    "left_knee": 4,
    "left_ankle": 5,
    "right_hip_yaw": 6,
    "right_hip_roll": 7,
    "right_hip_pitch": 8,
    "right_knee": 9,
    "right_ankle": 10,
    "torso": 11,
    "left_shoulder_pitch": 12,
    "left_shoulder_roll": 13,
    "left_shoulder_yaw": 14,
    "left_elbow": 15,
    "right_shoulder_pitch": 16,
    "right_shoulder_roll": 17,
    "right_shoulder_yaw": 18,
    "right_elbow": 19,
}

print(f"{'Joint':20s} | {'FK':30s} | {'Xanchor':30s} | {'Max Diff':8s}")
print("-" * 100)

max_diff_overall = 0.0
max_diff_joint = ""
sum_err = 0.0
count = 0

for joint_name, xanchor_idx in joint_to_xanchor_mapping.items():
    if joint_name in joint_positions:
        fk_pos = joint_positions[joint_name]
        xanchor_pos = data.xanchor[xanchor_idx]
        diff = np.abs(fk_pos - xanchor_pos)
        max_diff = np.max(diff)
        rmse = np.sqrt(np.mean((fk_pos - xanchor_pos) ** 2))
        
        sum_err += rmse
        count += 1
        
        if max_diff > max_diff_overall:
            max_diff_overall = max_diff
            max_diff_joint = joint_name
        
        fk_str = np.array2string(fk_pos, precision=3, separator=',', suppress_small=True)
        xanchor_str = np.array2string(xanchor_pos, precision=3, separator=',', suppress_small=True)
        
        print(f"{joint_name:20s} | {fk_str:30s} | {xanchor_str:30s} | {max_diff:8.6f}")

avg_rmse = (sum_err / max(count,1))
print(f"\nLargest difference: {max_diff_overall:.6f} at joint '{max_diff_joint}'")
print(f"Average RMSE: {avg_rmse:.6f}")
